# Cross-y analysis

In previous experiments, the fitted u only correspond to a single observation y. This notebook now tries to analyse how a certain feature could affect several observations simultaneously.

In [19]:
import sys
import os
import pickle
import heapq

sys.path.insert(0, os.path.abspath(".."))

import numpy as np

from sklearn.preprocessing import StandardScaler

from src.function_library import build_function_library,build_interaction_library, select_top_power_features, power_features, build_power_library
from src.evaluation import build_theta_from_model, visualize_sparse_prediction, analyze_feature_cmi, analyze_feature_importance
from src.sparse_interp import sparse_ee_interpretation, save_sparse_result, load_sparse_result, sparse_predict, refine_sparse_result

from npeet import entropy_estimators as ee
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from itertools import combinations

In [2]:
def ee_objective(coeffs, X, y):
    coeffs = np.asarray(coeffs, dtype=float)
    coeffs /= np.linalg.norm(coeffs)
    u = X @ coeffs
    return ee.mi(y, u)

def scipy_objective(coeffs, X, y):
    return -ee_objective(coeffs, X, y)

In [10]:
# loading data
data = np.load(r"../../data/final_data.npz")
print(data.files)

X_final = data["param"]

param_names = [
    "C_p",
    "Za_p",
    "R_p",
    "Emax_rv",
    "Emin_rv",
    "C_s",
    "Za_s",
    "R_s",
    "Emax_lv",
    "Emin_lv"
]

v_lv_final = data["v_lv"]
v_rv_final = data["v_rv"]

y_v_lv_max_final = np.max(v_lv_final, axis=1)
y_v_lv_min_final = np.min(v_lv_final, axis=1)

y_v_rv_max_final = np.max(v_rv_final, axis=1)
y_v_rv_min_final = np.min(v_rv_final, axis=1)

y_v_lv_mean_final = np.mean(v_lv_final, axis=1)
y_v_rv_mean_final = np.mean(v_rv_final, axis=1)

y_v_combined = np.column_stack([
    y_v_lv_max_final,
    y_v_lv_min_final,
    y_v_lv_mean_final,
    y_v_rv_max_final,
    y_v_rv_min_final,
    y_v_rv_mean_final,
])

y_v_lv_combined = np.column_stack([
    y_v_lv_max_final,
    y_v_lv_min_final,
    y_v_lv_mean_final,
])
y_v_rv_combined = np.column_stack([
    y_v_rv_max_final,
    y_v_rv_min_final,
    y_v_rv_mean_final,
])

y_v_max_combined = np.column_stack([
    y_v_lv_max_final,
    y_v_rv_max_final,
])
y_v_min_combined = np.column_stack([
    y_v_lv_min_final,
    y_v_rv_min_final,
])
y_v_mean_combined = np.column_stack([
    y_v_lv_mean_final,
    y_v_rv_mean_final,
])


['p_lv', 'p_rv', 'p_pa', 'v_lv', 'v_rv', 'q_av', 'q_mv', 'q_pv', 'param']


In [11]:
# load old sampling
with open("../notebook/final_data_split.pkl", "rb") as f:
    final_data_split = pickle.load(f)

final_train_idx = final_data_split["train_idx"]
final_test_idx = final_data_split["test_idx"]

X_final_train = X_final[final_train_idx]
X_final_test = X_final[final_test_idx]

y_v_lv_max_final_train = y_v_lv_max_final[final_train_idx]
y_v_lv_max_final_test = y_v_lv_max_final[final_test_idx]

y_v_lv_min_final_train = y_v_lv_min_final[final_train_idx]
y_v_lv_min_final_test = y_v_lv_min_final[final_test_idx]

y_v_rv_max_final_train = y_v_rv_max_final[final_train_idx]
y_v_rv_max_final_test = y_v_rv_max_final[final_test_idx]

y_v_rv_min_final_train = y_v_rv_min_final[final_train_idx]
y_v_rv_min_final_test = y_v_rv_min_final[final_test_idx]

y_v_lv_mean_final_train = y_v_lv_mean_final[final_train_idx]
y_v_rv_mean_final_train = y_v_rv_mean_final[final_train_idx]

y_v_lv_mean_final_test = y_v_lv_mean_final[final_test_idx]
y_v_rv_mean_final_test = y_v_rv_mean_final[final_test_idx]

# LV combined
y_v_lv_combined_train = y_v_lv_combined[final_train_idx]
y_v_lv_combined_test = y_v_lv_combined[final_test_idx]

# RV combined
y_v_rv_combined_train = y_v_rv_combined[final_train_idx]
y_v_rv_combined_test = y_v_rv_combined[final_test_idx]

# Max combined
y_v_max_combined_train = y_v_max_combined[final_train_idx]
y_v_max_combined_test = y_v_max_combined[final_test_idx]

# Min combined
y_v_min_combined_train = y_v_min_combined[final_train_idx]
y_v_min_combined_test = y_v_min_combined[final_test_idx]

# Mean combined
y_v_mean_combined_train = y_v_mean_combined[final_train_idx]
y_v_mean_combined_test = y_v_mean_combined[final_test_idx]

# All combined
y_v_combined_train = y_v_combined[final_train_idx]
y_v_combined_test = y_v_combined[final_test_idx]

In [14]:
# original library
Theta_v, feature_names_v = build_function_library(
    X_final_train,
    param_names
)


# new library
# Theta_v, feature_names_v = (
#     build_interaction_library(
#         X_final_train,
#         param_names
#     )
# )

In [30]:
def analyze_joint_mi(
    Theta,
    feature_names,
    Y,
    k=1,
    top_n=100,
):
    Theta = np.asarray(Theta)
    Y = np.asarray(Y)

    top_results = []

    for indices in combinations(range(Theta.shape[1]), k):

        X_combined = Theta[:, indices]

        mi = ee.mi(
            X_combined,
            Y
        )

        # Store only top_n results
        if len(top_results) < top_n:

            heapq.heappush(
                top_results,
                (mi, indices)
            )

        elif mi > top_results[0][0]:

            heapq.heapreplace(
                top_results,
                (mi, indices)
            )

    # Sort only the top_n results
    top_results.sort(
        key=lambda x: x[0],
        reverse=True
    )

    # Convert indices to feature names
    results = []

    for mi, indices in top_results:

        feature_combination = tuple(
            feature_names[i]
            for i in indices
        )

        results.append({
            "features": feature_combination,
            "mi": mi
        })

    return results

def print_joint_mi_results(
    results_dict,
    top_n=50,
    orderby=None,
):
    combinations_list = []

    for results in results_dict.values():
        for result in results:
            feature_combination = result["features"]

            if feature_combination not in combinations_list:
                combinations_list.append(feature_combination)

    # Store MI values
    mi_table = {}

    for feature_combination in combinations_list:
        mi_table[feature_combination] = {}

    for target_name, results in results_dict.items():

        for result in results:

            feature_combination = result["features"]
            mi = result["mi"]

            mi_table[feature_combination][target_name] = mi

    # Print header
    target_names = list(results_dict.keys())

    header = f"{'Features':50s}"

    for target_name in target_names:
        header += f"{target_name:>12s}"

    print(header)
    print("-" * len(header))

    # Determine sorting target
    if orderby is None:
        orderby = target_names[0]

    if orderby not in target_names:
        raise ValueError(
            f"Unknown orderby '{orderby}'. "
            f"Available targets: {target_names}"
        )

    # Sort by selected target
    combinations_list.sort(
        key=lambda combination:
            mi_table[combination].get(
                orderby,
                float("-inf")
            ),
        reverse=True
    )

    # Only display top N
    combinations_to_print = combinations_list[:top_n]

    # Print rows
    for feature_combination in combinations_to_print:

        feature_string = " & ".join(
            feature_combination
        )

        row = f"{feature_string:50s}"

        for target_name in target_names:

            mi = mi_table[feature_combination].get(
                target_name,
                float("nan")
            )

            row += f"{mi:12.6f}"

        print(row)

    return mi_table

def save_joint_mi_results(
    results,
    filename,
):
    with open(filename, "wb") as f:
        pickle.dump(results, f)


def load_joint_mi_results(
    filename,
):
    with open(filename, "rb") as f:
        results = pickle.load(f)

    return results

In [36]:
results_v_lv = analyze_joint_mi(
    Theta_v,
    feature_names_v,
    y_v_lv_combined_train,
    k=4
)
results_v_rv = analyze_joint_mi(
    Theta_v,
    feature_names_v,
    y_v_rv_combined_train,
    k=4
)
results_v_max = analyze_joint_mi(
    Theta_v,
    feature_names_v,
    y_v_max_combined_train,
    k=4
)
results_v_min = analyze_joint_mi(
    Theta_v,
    feature_names_v,
    y_v_min_combined_train,
    k=4
)
results_v_mean = analyze_joint_mi(
    Theta_v,
    feature_names_v,
    y_v_mean_combined_train,
    k=4
)
results_v = analyze_joint_mi(
    Theta_v,
    feature_names_v,
    y_v_combined_train,
    k=4
)

In [37]:
# the order of dict decides the table's sorting order, sorted by the first term
results_dict = {
    "All": results_v,
    "LV": results_v_lv,
    "RV": results_v_rv,
    "Max": results_v_max,
    "Min": results_v_min,
    "Mean": results_v_mean,
}

save_joint_mi_results(
    results_dict,
    "../results/joint_mi_k4.pkl"
)

In [51]:
results_dict = load_joint_mi_results(
    "../results/joint_mi_k4.pkl"
)

mi_table = print_joint_mi_results(
    results_dict,
    20,
    "RV"
)

Features                                                   All          LV          RV         Max         Min        Mean
--------------------------------------------------------------------------------------------------------------------------
C_s^2 & Emax_lv^2 & C_p*R_s & R_p*Emin_rv             1.560402         nan    1.181335         nan         nan         nan
Emax_lv^2 & log(Emax_lv) & C_p*R_s & R_p*Emin_rv      1.560959         nan    1.181165         nan         nan         nan
Emax_lv^2 & C_p*R_s & R_p*Emin_rv & Emax_rv*Emin_lv    1.559210         nan    1.181139    1.575591         nan         nan
Emax_lv^2 & C_p*R_s & R_p*Emin_rv & C_s*Emax_lv       1.561571         nan    1.180974         nan         nan         nan
Emax_lv^2 & C_p*R_s & C_p*Emin_lv & R_p*Emin_rv       1.560024         nan    1.180937    1.576661         nan         nan
C_p & Emax_lv^2 & C_p*R_s & R_p*Emin_rv               1.560263         nan    1.180667         nan         nan         nan
Emax_lv^2 & log

In [39]:
Theta_new_v, feature_names_new_v = (
    build_interaction_library(
        X_final_train,
        param_names
    )
)

In [40]:
results_v_lv = analyze_joint_mi(
    Theta_new_v,
    feature_names_new_v,
    y_v_lv_combined_train,
    k=4
)
results_v_rv = analyze_joint_mi(
    Theta_new_v,
    feature_names_new_v,
    y_v_rv_combined_train,
    k=4
)
results_v_max = analyze_joint_mi(
    Theta_new_v,
    feature_names_new_v,
    y_v_max_combined_train,
    k=4
)
results_v_min = analyze_joint_mi(
    Theta_new_v,
    feature_names_new_v,
    y_v_min_combined_train,
    k=4
)
results_v_mean = analyze_joint_mi(
    Theta_new_v,
    feature_names_new_v,
    y_v_mean_combined_train,
    k=4
)
results_v = analyze_joint_mi(
    Theta_new_v,
    feature_names_new_v,
    y_v_combined_train,
    k=4
)

In [41]:
results_dict = {
    "All": results_v,
    "LV": results_v_lv,
    "RV": results_v_rv,
    "Max": results_v_max,
    "Min": results_v_min,
    "Mean": results_v_mean,
}

save_joint_mi_results(
    results_dict,
    "../results/joint_mi_new_k4.pkl"
)

In [50]:
results_dict = load_joint_mi_results(
    "../results/joint_mi_new_k4.pkl"
)

mi_table = print_joint_mi_results(
    results_dict,
    20,
    "RV"
)

Features                                                   All          LV          RV         Max         Min        Mean
--------------------------------------------------------------------------------------------------------------------------
Emax_lv+Emax_rv & Emax_lv/Emax_rv & Emax_rv/Emin_rv & R_s/R_p         nan         nan    1.091327         nan         nan         nan
Emax_lv-Emin_lv & Emax_lv/Emax_rv & Emax_rv/Emin_rv & R_s/R_p         nan         nan    1.086858         nan         nan         nan
Emax_lv/Emax_rv & Emax_lv+Emin_rv & Emax_rv/Emin_rv & R_s/R_p         nan         nan    1.086366         nan         nan         nan
Emax_lv/Emax_rv & Emax_lv-Emin_rv & Emax_rv/Emin_rv & R_s/R_p         nan         nan    1.085889         nan         nan         nan
Emax_lv & Emax_lv/Emax_rv & Emax_rv/Emin_rv & R_s/R_p         nan         nan    1.085783         nan         nan         nan
Emax_lv+Emin_lv & Emax_lv/Emax_rv & Emax_rv/Emin_rv & R_s/R_p         nan         nan    1.0